# Lab 05 — Shuffle and GroupBy

**Tujuan:** Melihat redistribusi data, stage boundary, dan pengaruh shuffle partition.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 05")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

In [ ]:
df = spark.range(0, 2_000_000, 1, 4)
data = df.withColumn("group_id", F.col("id") % 100)
summary = data.groupBy("group_id").count()
summary.explain("formatted")

**Prediksi:** mengapa semua baris dengan `group_id` sama harus bertemu pada pengurangan akhir?

In [ ]:
summary.orderBy("group_id").show(10)

## Control shuffle partitions

Untuk membandingkan, matikan AQE sementara agar bentuk shuffle lebih mudah diprediksi; kembalikan setelah eksperimen.

In [ ]:
old_aqe = spark.conf.get("spark.sql.adaptive.enabled")
old_shuffle = spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "4")
summary_4 = data.groupBy("group_id").count()
summary_4.explain("formatted")
summary_4.collect()
spark.conf.set("spark.sql.shuffle.partitions", "12")
summary_12 = data.groupBy("group_id").count()
summary_12.explain("formatted")
summary_12.collect()
spark.conf.set("spark.sql.shuffle.partitions", old_shuffle)
spark.conf.set("spark.sql.adaptive.enabled", old_aqe)

**Tugas:** bandingkan jumlah reduce tasks, shuffle metrics, dan waktu. Jalankan beberapa kali sebelum membuat kesimpulan.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.